# Paper 2 topic model fitting, selection, and stability

Fit one shared BERTopic space on Paper 2 article passages and an equally sized, deterministic comment sample. Comments are drawn from the eligible analysis corpus, round-robin across stories in hashed order so large discussions do not dominate the sample. This is corpus-level measurement, not held-out predictive evaluation.

BERTopic already uses class-based TF-IDF (c-TF-IDF). German/English stopwords are removed by the representation vectorizer, and `reduce_frequent_words=True` downweights frequent terms. Embeddings retain the original text. Automatic topic reduction remains enabled.

BERTopic outlier labels are retained as `-1`; no embedding-based outlier reassignment is applied. Raw assignments, coverage, and outlier counts are reported directly so topic quality is not conflated with threshold-dependent imputation.


In [ ]:
from dataclasses import asdict, replace
from itertools import product
from pathlib import Path
import gc
import json
import subprocess
import sys
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from commentgap_analysis.topic_modeling import TopicModelConfig
from commentgap_analysis.topic_diagnostics import (
    load_diagnostic_corpus, corpus_signature, diagnostic_fit, stability_pairs,
    load_search_artifacts, search_cache_matches,
)
from commentgap_analysis.topic_diagnostics import pooled_stability_pairs
from commentgap_analysis.topic_runs import list_runs, resolve_run, select_run, write_json

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'commentgap_analysis').exists() and (REPO_ROOT.parent / 'commentgap_analysis').exists():
    REPO_ROOT = REPO_ROOT.parent
DATA_ROOT = REPO_ROOT / 'data/scrape_2025'
EMBEDDING_STORE = REPO_ROOT / 'model_output/selection_2025/embeddings/model=BAAI__bge-m3--d790e737/build=de5b3016fb2f-010a7cc75a88'
SPLIT_PATH = REPO_ROOT / 'model_output/selection_2025/model_data/master_article_split.parquet'
ANALYSIS_COMMENTS_PATH = REPO_ROOT / 'model_output/selection_2025/paper2/analysis_comments.parquet'
RUNS_ROOT = REPO_ROOT / 'model_output/selection_2025/paper2_topic_runs'
DIAGNOSTIC_ROOT = REPO_ROOT / 'model_output/selection_2025/paper2_topic_diagnostics'
YEAR, BATCH_SIZE, WORKERS = 2025, 10_000, 1
BASE_CONFIG = TopicModelConfig(
    representation_stopwords=True, reduce_frequent_words=True, outlier_threshold=None,
)
FIT_ROLE = 'paper2_test'

# 60 mixed-corpus parameter setups, three seeds each: 180 fits.
FIT_CORPORA = ['articles_comments']
MIN_CLUSTER_SIZES = [5, 10, 15]
N_NEIGHBORS = [5, 10, 15, 20]
MIN_SAMPLES = [1, 3, 5, 10, 15]
STABILITY_SEEDS = [2025, 2026, 2027]
FIT_WORKERS = 3  # Concurrent seed fits per setup; use 1 for serial execution.
RUN_SEARCH = False  # Set True only when refitting the full parameter search.
RUN_FINAL = False  # Enable when you run the final fit and Paper 2 labeling section.
# Set to a completed search directory (or its directory name) to load it regardless of current cache metadata.
SEARCH_ROOT_OVERRIDE = '7cf9d9ba0750d91b'  # Latest complete search run.

# Initial final configuration; review after the new Paper 2 search: mixed article/comment fit, mcs=10, nn=10, ms=3, seed=2026.
FINAL_CONFIG = replace(
    BASE_CONFIG, random_state=2026, umap_n_neighbors=10,
    hdbscan_min_cluster_size=15, hdbscan_min_samples=5,
)
FINAL_FIT_CORPUS = 'articles_comments'
RUN_TAG = ''
print('Search fits:', len(FIT_CORPORA) * len(MIN_CLUSTER_SIZES) * len(N_NEIGHBORS) * len(MIN_SAMPLES) * len(STABILITY_SEEDS))
print('Concurrent seed fits per setup:', FIT_WORKERS)
print('Current final configuration (review after search):', asdict(FINAL_CONFIG))

## 1. Load the Paper 2 fitting corpus

Use all Paper 2 article passages and exactly the same number of eligible comments. Both document types contribute to topic discovery. Diagnostics report in-sample coverage from raw assignments, retaining rejected documents as `-1`; they do not estimate generalisation. Seed agreement measures sensitivity to UMAP randomness, not sensitivity to the story sample. Freeze selection before comparing ranking effects.


In [ ]:
split = pd.read_parquet(SPLIT_PATH, columns=['story_id', 'split_role'])
display(split.groupby('split_role').size().rename('stories'))
print('Prepared final-transform comment rows:', pq.ParquetFile(ANALYSIS_COMMENTS_PATH).metadata.num_rows)
if RUN_SEARCH:
    corpus = load_diagnostic_corpus(
        DATA_ROOT, SPLIT_PATH, EMBEDDING_STORE, YEAR, fit_role=FIT_ROLE, analysis_comments_path=ANALYSIS_COMMENTS_PATH,
    )
    expected_diagnostic_keys = {(FIT_ROLE, 'article_passage'), (FIT_ROLE, 'comment')}
    if set(corpus) != expected_diagnostic_keys:
        raise ValueError(f'Diagnostic corpus must contain Paper 2 articles/comments only; got {sorted(corpus)}')
    diagnostic_signature = corpus_signature(corpus)
    # Persist the exact current sample for a later mixed final fit.
    FIT_COMMENT_DOCUMENTS_PATH = DIAGNOSTIC_ROOT / 'corpora' / diagnostic_signature / 'fit_comment_documents.parquet'
    FIT_COMMENT_DOCUMENTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    corpus[(FIT_ROLE, 'comment')][0].to_parquet(FIT_COMMENT_DOCUMENTS_PATH, index=False)


## 2. Joint parameter and stability search

Cross `min_cluster_size=[5,10,15]`, `n_neighbors=[5,10,15,20]`, and `min_samples=[1,3,5,10,15]` across seeds 2025–2027: **180 mixed-corpus fits**. All fits use the exact same Paper 2 passages and size-matched comment sample. Matching fits are cached; corpus, configuration, code, and software identify caches.

Coverage and stability use the raw BERTopic assignments, retaining rejected documents as `-1`. `raw_coverage_pct` and `outlier_documents` in seed summaries show how much of the corpus receives a substantive topic.

Common AMI conditions on assignment in both seeds; union retains inclusion disagreements; all includes the shared outlier group. Pooled scores combine article and comment rows. Shared rejection can inflate all-document agreement. Pairwise comparisons are not independent observations, and degenerate comparisons are missing rather than perfect scores.


In [ ]:
if RUN_SEARCH:
    from commentgap_analysis.topic_runs import setup_id
    grid = list(product(FIT_CORPORA, MIN_CLUSTER_SIZES, N_NEIGHBORS, MIN_SAMPLES))
    if not grid or len(grid) != len(set(grid)):
        raise ValueError('Use a nonempty grid of distinct parameter combinations')
    if len(STABILITY_SEEDS) < 3 or len(set(STABILITY_SEEDS)) != len(STABILITY_SEEDS):
        raise ValueError('Use at least three distinct seeds')
    if int(FIT_WORKERS) < 1:
        raise ValueError('FIT_WORKERS must be at least 1')
    search_setup = {
        'base_configuration': asdict(BASE_CONFIG), 'min_cluster_sizes': MIN_CLUSTER_SIZES,
        'n_neighbors': N_NEIGHBORS, 'min_samples': MIN_SAMPLES, 'fit_corpora': FIT_CORPORA,
        'fit_workers': int(FIT_WORKERS), 'fit_role': FIT_ROLE,
        'seeds': STABILITY_SEEDS, 'corpus_sha256': diagnostic_signature,
    }
    search_setup['code_hashes'] = {
        name: __import__('hashlib').sha256((REPO_ROOT / 'commentgap_analysis' / name).read_bytes()).hexdigest()
        for name in ('topic_diagnostics.py', 'topic_modeling.py')
    }
    search_root = DIAGNOSTIC_ROOT / 'searches' / setup_id(search_setup)
    write_json(search_root / 'configuration.json', search_setup)
    all_seed_summaries, all_pairs, all_pooled_pairs, inclusion_rows = [], [], [], []
    for setup_number, (fit_corpus, min_cluster_size, n_neighbors, min_samples) in enumerate(grid, 1):
        config = replace(BASE_CONFIG, hdbscan_min_cluster_size=min_cluster_size,
                         hdbscan_min_samples=min_samples, umap_n_neighbors=n_neighbors)
        setup_name = f'{fit_corpus}_mcs{min_cluster_size}_nn{n_neighbors}_ms{min_samples}'
        print(f'Setup {setup_number}/{len(grid)}: {setup_name}', flush=True)
        from concurrent.futures import ThreadPoolExecutor, as_completed
        def _run_seed(seed):
            print(f'  START {setup_name}, seed={seed}', flush=True)
            result = diagnostic_fit(
                corpus, replace(config, random_state=seed), DIAGNOSTIC_ROOT,
                diagnostic_signature, batch_size=BATCH_SIZE, fit_corpus=fit_corpus, fit_role=FIT_ROLE,
            )
            print(f'  COMPLETE {setup_name}, seed={seed}', flush=True)
            return result
        seed_results = {}
        with ThreadPoolExecutor(max_workers=int(FIT_WORKERS),
                                thread_name_prefix='topic-fit') as executor:
            futures = {executor.submit(_run_seed, seed): seed for seed in STABILITY_SEEDS}
            for future in as_completed(futures):
                seed_results[futures[future]] = future.result()
        seed_summaries, seed_assignments = [], []
        for seed in STABILITY_SEEDS:
            summary, assignments = seed_results[seed]
            if set(summary.split_role.unique()) != {FIT_ROLE} or set(assignments.split_role.unique()) != {FIT_ROLE}:
                raise ValueError('Search diagnostics must contain Paper 2 documents only')
            seed_summaries.append(summary)
            seed_assignments.append(
                assignments.loc[assignments.split_role.eq(FIT_ROLE)]
                .drop(columns='split_role').set_index(['doc_id', 'story_id', 'doc_type'])
                .rename(columns={
                    'topic_assignment': f'seed_{seed}',
                    'raw_topic_assignment': f'raw_seed_{seed}',
                    'outlier_reassigned': f'outlier_reassigned_seed_{seed}',
                })
            )
        aligned = pd.concat(seed_assignments, axis=1).reset_index()
        label_columns = [f'seed_{seed}' for seed in STABILITY_SEEDS]
        pairs = stability_pairs(aligned, label_columns).assign(
            min_cluster_size=min_cluster_size, n_neighbors=n_neighbors, min_samples=min_samples,
            fit_corpus=fit_corpus,
        )
        pooled_pairs = pooled_stability_pairs(aligned, label_columns).assign(
            min_cluster_size=min_cluster_size, n_neighbors=n_neighbors, min_samples=min_samples,
            fit_corpus=fit_corpus,
        )
        seed_summary = pd.concat(seed_summaries, ignore_index=True)
        setup_root = search_root / setup_name
        write_json(setup_root / 'configuration.json', {
            'configuration': asdict(config), 'seeds': STABILITY_SEEDS, 'fit_corpus': fit_corpus,
            'diagnostic_run_ids': seed_summary.run_id.drop_duplicates().tolist(),
        })
        pairs.to_csv(setup_root / 'pairs.csv', index=False)
        pooled_pairs.to_csv(setup_root / 'pooled_pairs.csv', index=False)
        seed_summary.to_csv(setup_root / 'seeds.csv', index=False)
        aligned.to_parquet(setup_root / 'assignments.parquet', index=False)
        for doc_type, documents in aligned.groupby('doc_type'):
            inclusion_rows.append({
                'min_cluster_size': min_cluster_size, 'n_neighbors': n_neighbors,
                'min_samples': min_samples, 'doc_type': doc_type, 'fit_corpus': fit_corpus,
                'all_seed_coverage_pct': 100 * documents[label_columns].ne(-1).all(axis=1).mean(),
            })
        all_seed_summaries.append(seed_summary)
        all_pairs.append(pairs)
        all_pooled_pairs.append(pooled_pairs)
        display(seed_summary[['seed', 'fit_corpus', 'doc_type', 'label_source', 'fitted_substantive_topics',
                              'assigned_documents', 'raw_coverage_pct', 'coverage_pct', 'reassigned_documents']].round(3))

    search_seeds = pd.concat(all_seed_summaries, ignore_index=True)
    search_pairs = pd.concat(all_pairs, ignore_index=True)
    search_pooled_pairs = pd.concat(all_pooled_pairs, ignore_index=True)
    group_columns = ['fit_corpus', 'min_cluster_size', 'n_neighbors', 'min_samples', 'doc_type']
    coverage_report = search_seeds.groupby(group_columns)[
        ['coverage_pct', 'assigned_documents', 'fitted_substantive_topics']
    ].agg(['min', 'median', 'max'])
    stability_report = search_pairs.groupby(group_columns)[
        ['ami_common', 'ami_union', 'ami_all',
         'assigned_jaccard', 'common_coverage_pct']
    ].agg(['min', 'median', 'max', 'count'])
    pooled_group_columns = ['fit_corpus', 'min_cluster_size', 'n_neighbors', 'min_samples']
    pooled_stability_report = search_pooled_pairs.groupby(pooled_group_columns)[
        ['ami_common', 'ami_union', 'ami_all']
    ].agg(['min', 'median', 'max', 'count'])
    for report in (coverage_report, stability_report):
        report.columns = ['_'.join(column) for column in report.columns]
    pooled_stability_report.columns = [
        f'{metric}_pooled_{stat}' for metric, stat in pooled_stability_report.columns
    ]
    search_report = coverage_report.join(stability_report).reset_index().merge(
        pd.DataFrame(inclusion_rows), on=group_columns, validate='one_to_one',
    ).merge(
        pooled_stability_report.reset_index(), on=pooled_group_columns, validate='many_to_one',
    )
    search_report['coverage_range_pp'] = search_report.coverage_pct_max - search_report.coverage_pct_min
    search_seeds.to_csv(search_root / 'seeds.csv', index=False)
    search_pairs.to_csv(search_root / 'pairs.csv', index=False)
    search_pooled_pairs.to_csv(search_root / 'pooled_pairs.csv', index=False)
    search_report.to_csv(search_root / 'summary.csv', index=False)
    print('Combined Paper 2 search (full metrics saved to summary.csv):')
    display(search_report[group_columns + [
        'coverage_pct_min', 'coverage_pct_median', 'coverage_range_pp',
        'fitted_substantive_topics_median', 'ami_common_min', 'ami_common_median',
        'ami_union_median', 'assigned_jaccard_median',
        'ami_all_min', 'ami_all_median',
        'ami_common_pooled_median',
        'ami_union_pooled_median',
        'ami_all_pooled_median',
        'all_seed_coverage_pct',
    ]].round(3))
    print('Search artifacts:', search_root)

In [ ]:
# Load completed search artifacts without loading embeddings or fitting models.
cache_root = DIAGNOSTIC_ROOT / 'searches'
if SEARCH_ROOT_OVERRIDE is not None:
    search_root = Path(SEARCH_ROOT_OVERRIDE)
    if not search_root.is_absolute():
        search_root = cache_root / search_root
    print(f'Loading explicitly selected search: {search_root}')
else:
    expected_cache = {
        'fit_corpora': FIT_CORPORA,
        'fit_role': FIT_ROLE,
        'base_configuration': json.loads(json.dumps(asdict(BASE_CONFIG))),
        'code_hashes': {name: __import__('hashlib').sha256((REPO_ROOT / 'commentgap_analysis' / name).read_bytes()).hexdigest()
                        for name in ('topic_diagnostics.py', 'topic_modeling.py')},
        'min_cluster_sizes': MIN_CLUSTER_SIZES,
        'n_neighbors': N_NEIGHBORS,
        'min_samples': MIN_SAMPLES,
        'seeds': STABILITY_SEEDS,
    }
    cache_candidates = [path for path in (cache_root.iterdir() if cache_root.exists() else [])
                       if path.is_dir() and search_cache_matches(path, expected_cache)]
    if not cache_candidates:
        raise FileNotFoundError('No completed cache matches the current search grid.')
    search_root = max(cache_candidates, key=lambda path: (path / 'summary.csv').stat().st_mtime)
loaded_search = load_search_artifacts(search_root, diagnostic_root=DIAGNOSTIC_ROOT)
search_root = loaded_search['search_root']
cache_configuration = loaded_search['cache_configuration']
diagnostic_signature = loaded_search['diagnostic_signature']
search_seeds = loaded_search['search_seeds']
search_pairs = loaded_search['search_pairs']
search_pooled_pairs = loaded_search['search_pooled_pairs']
search_report = loaded_search['search_report']
if loaded_search['fit_comment_documents_path'] is not None:
    FIT_COMMENT_DOCUMENTS_PATH = loaded_search['fit_comment_documents_path']
elif diagnostic_signature:
    print('Warning: the selected search has no saved fit comment sample; final fitting may need one.')
print(f'Loaded cached search: {search_root.name}')
print(f'Configurations: {len(search_report)}; seed summaries: {len(search_seeds)}')
print(f'Final-fit comment sample: {FIT_COMMENT_DOCUMENTS_PATH}')

### Coverage and pooled common stability

Compare mixed Paper 2 fits on article coverage, comment coverage, and pooled common AMI using raw assignments. The Pareto frontier maximises all three. Topic counts and small-topic document counts are descriptive. Review coverage and outlier counts in `search_seeds` and inspect topic examples before choosing a configuration.


In [ ]:
from commentgap_analysis.topic_plotting import plot_topic_search_metrics

# Reuse the completed search; this cell does not fit any models.
if 'search_report' not in globals():
    raise RuntimeError('Run the search cell, or load its summary.csv as search_report first.')
if 'search_root' not in globals():
    raise RuntimeError('The search root is not available; rerun the search setup cell first.')

plot_data, frontier = plot_topic_search_metrics(
    search_report=search_report,
    search_seeds=search_seeds,
    search_root=search_root,
    stability_seeds=STABILITY_SEEDS,
    final_configuration=FINAL_CONFIG,
    final_fit_corpus=FINAL_FIT_CORPUS,
)
frontier_columns = [
    'configuration', 'article_coverage', 'comment_coverage',
    'raw_pooled_common_ami', 'topics',
    'docs_in_topics_le5', 'docs_in_topics_le10',
]
display(
    frontier[frontier_columns].rename(columns={
        'article_coverage': 'article coverage (%)',
        'comment_coverage': 'comment coverage (%)',
        'raw_pooled_common_ami': 'raw pooled common AMI',
        'docs_in_topics_le5': 'docs in topics <=5',
        'docs_in_topics_le10': 'docs in topics <=10',
    }).round({
        'article coverage (%)': 3,
        'comment coverage (%)': 3,
        'raw pooled common AMI': 4,
    })
)


## 3. Final fit and transform

Review `FINAL_CONFIG` in the configuration cell, then enable `RUN_FINAL`. The exact configuration is passed to the runner. Final outputs live at `paper2_topic_runs/<setup-id>/`, with a readable parameter/seed prefix and a hash covering configuration, inputs, code, software versions, execution settings, and optional replicate tag.

`run.json` records setup and completion; existing model/run manifests and transformation checkpoints remain inside that directory. Identical setups reuse completed runs or resume unfinished transformations. Different setups have separate artifacts. Input tracking hashes the split/comment files and embedding manifest and inventories parquet shard paths, sizes, and modification times (the shard inventory is not a full content checksum). Keep the source data and embedding build immutable. Change `RUN_TAG` to record another execution of the same setup.

Each run's downstream calculations and figures will live in its own `analysis/` directory. Legacy artifacts remain at their existing paths and can still be chosen explicitly in notebooks 15 and 16.

`FINAL_FIT_CORPUS="articles_comments"` passes the exact saved Paper 2 comment sample to the runner. Its path and checksum are recorded in the final run identity. When running this section separately, set `FIT_COMMENT_DOCUMENTS_PATH` to the sample exported by the desired search.


In [ ]:
# Release Paper 2 matrices before launching the final subprocess.
if 'corpus' in globals():
    del corpus
    gc.collect()

if RUN_FINAL:
    config_path = DIAGNOSTIC_ROOT / 'requested_final_config.json'
    write_json(config_path, asdict(FINAL_CONFIG))
    command = [
        sys.executable, '-m', 'scripts.run_topic_model_fit',
        '--repo-root', str(REPO_ROOT), '--data-root', str(DATA_ROOT),
        '--fit-role', FIT_ROLE, '--split', str(SPLIT_PATH), '--analysis-comments', str(ANALYSIS_COMMENTS_PATH),
        '--runs-root', str(RUNS_ROOT), '--embedding-store', str(EMBEDDING_STORE),
        '--model-config', str(config_path), '--run-tag', RUN_TAG,
        '--year', str(YEAR), '--batch-size', str(BATCH_SIZE), '--workers', str(WORKERS),
    ]
    if 'FIT_COMMENT_DOCUMENTS_PATH' not in globals():
        raise RuntimeError('Load or run the Paper 2 search to obtain its exact comment sample.')
    command.extend(['--fit-comment-documents', str(FIT_COMMENT_DOCUMENTS_PATH)])
    subprocess.run(command, cwd=REPO_ROOT, check=True)
display(pd.DataFrame(list_runs(RUNS_ROOT)))

## 4. Select the run for downstream notebooks

Copy a completed `run_id` from the table into `SELECT_RUN_ID`. Executing this cell saves a shared selection for notebooks 15 and 16. Selection is explicit: finishing a new fit never silently switches the analysis. Downstream notebooks can instead pin an individual run ID for reproducibility. No fit is launched by this cell.

In [ ]:
SELECT_RUN_ID = 'mcs15_nn10_ms5_seed2026_1e1b8fce5deb3e58'
if SELECT_RUN_ID is not None:
    TOPIC_ROOT = resolve_run(RUNS_ROOT, SELECT_RUN_ID)
elif (RUNS_ROOT / 'selected_run.json').exists():
    TOPIC_ROOT = resolve_run(RUNS_ROOT)
else:
    TOPIC_ROOT = None
    print('No final run selected yet.')
if TOPIC_ROOT is not None:
    metadata = json.loads((TOPIC_ROOT / 'topic_model_run_metadata.json').read_text())
    if metadata['inputs']['fit_scope'] != FIT_ROLE or metadata['configuration'] != json.loads(json.dumps(asdict(FINAL_CONFIG))):
        raise ValueError('Select a completed run matching the new Paper 2 configuration.')
    if SELECT_RUN_ID is not None:
        TOPIC_ROOT = select_run(RUNS_ROOT, SELECT_RUN_ID)
    print('Selected topic artifacts:', TOPIC_ROOT)
    display(json.loads((TOPIC_ROOT / 'topic_model_run_metadata.json').read_text())['counts'])
    memberships = pq.ParquetFile(TOPIC_ROOT / 'document_topic_memberships.parquet')
    print('Membership rows:', memberships.metadata.num_rows)
    preview = next(memberships.iter_batches(batch_size=5), None)
    if preview is not None:
        preview = preview.to_pandas()
        preview_columns = [column for column in [
            'doc_id', 'story_id', 'comment_id', 'doc_type',
            'topic_assignment', 'valid_topic',
        ] if column in preview.columns]
        display(preview.loc[:, preview_columns])
    display(pd.read_parquet(TOPIC_ROOT / 'article_topic_coverage.parquet').head())

## 5. Topic terms and raw coverage

Topic terms come directly from the stopword-filtered c-TF-IDF representation; no display-only filtering is applied. Final coverage uses one consistent transform rule for every Paper 2 document, including those in the fitting sample; rejected documents remain unassigned. The final coverage table reports article and full-comment coverage; a separate comment table reports the exact fit comment sample after transform alongside all eligible comments after transform. This differs from fitted-label diagnostics above. Both are corpus measurement diagnostics, not train/test evaluation. Review outlier counts and coverage by story before interpreting ranking comparisons.


In [ ]:
if TOPIC_ROOT is None:
    raise RuntimeError('Select a completed Paper 2 run first.')
from commentgap_analysis.topic_modeling import load_topic_model

topic_bundle = load_topic_model(TOPIC_ROOT / 'topic_model.joblib')
topic_model = topic_bundle.model
assignments = pd.read_parquet(
    TOPIC_ROOT / 'document_topic_memberships.parquet',
    columns=['doc_id', 'story_id', 'doc_type', 'topic_assignment', 'raw_topic_assignment', 'outlier_reassigned', 'valid_topic'],
)
run_metadata = json.loads((TOPIC_ROOT / 'topic_model_run_metadata.json').read_text())
fit_comment_source = run_metadata['inputs'].get('fit_comment_documents')
if not fit_comment_source:
    raise ValueError('The selected run does not record its fit comment sample.')
fit_comment_ids = set(pd.read_parquet(fit_comment_source['path'], columns=['doc_id'])['doc_id'].astype(str))
assignments['is_fit_comment'] = (
    assignments['doc_type'].eq('comment') & assignments['doc_id'].astype(str).isin(fit_comment_ids)
)
if assignments.loc[assignments['is_fit_comment'], 'doc_id'].nunique() != len(fit_comment_ids):
    raise ValueError('The final transform output is missing documents from the fit comment sample.')
assignments['raw_valid'] = assignments.raw_topic_assignment.ne(-1)
coverage = assignments.groupby('doc_type').agg(
    n_documents=('valid_topic', 'size'),
    raw_assigned=('raw_valid', 'sum'),
    assigned=('valid_topic', 'sum'),
    reassigned=('outlier_reassigned', 'sum'),
).reset_index()
coverage['raw_coverage_pct'] = 100 * coverage.raw_assigned / coverage.n_documents
coverage['coverage_pct'] = 100 * coverage.assigned / coverage.n_documents
comment_assignments = assignments.loc[assignments['doc_type'].eq('comment')].copy()
comment_coverage = pd.concat([
    comment_assignments.assign(coverage_scope='all eligible comments after transform'),
    comment_assignments.loc[comment_assignments['is_fit_comment']].assign(
        coverage_scope='fit comment sample after transform'
    ),
]).groupby('coverage_scope').agg(
    n_documents=('valid_topic', 'size'),
    raw_assigned=('raw_valid', 'sum'),
    assigned=('valid_topic', 'sum'),
    reassigned=('outlier_reassigned', 'sum'),
).reset_index()
comment_coverage['raw_coverage_pct'] = 100 * comment_coverage.raw_assigned / comment_coverage.n_documents
comment_coverage['coverage_pct'] = 100 * comment_coverage.assigned / comment_coverage.n_documents

topic_summary = topic_model.get_topic_info()
topic_summary = topic_summary.loc[topic_summary.Topic.ne(-1)].copy()
topic_summary['top_terms'] = topic_summary.Topic.map(
    lambda topic: ' | '.join(term for term, _ in (topic_model.get_topic(int(topic)) or [])[:10])
)
topic_summary = topic_summary.rename(columns={'Count': 'n_fit_documents'})
display(topic_summary.head(20))
display(coverage.round(3))
display(comment_coverage.round(3))

story_coverage = assignments.groupby(['story_id', 'doc_type']).agg(
    n_documents=('valid_topic', 'size'), raw_coverage=('raw_valid', 'mean'),
    coverage=('valid_topic', 'mean'), reassigned=('outlier_reassigned', 'sum'),
).reset_index()
analysis_root = TOPIC_ROOT / 'analysis'
analysis_root.mkdir(parents=True, exist_ok=True)
topic_summary.to_csv(analysis_root / 'final_topic_summary.csv', index=False)
coverage.to_csv(analysis_root / 'final_document_coverage.csv', index=False)
comment_coverage.to_csv(analysis_root / 'final_comment_coverage.csv', index=False)
story_coverage.to_csv(analysis_root / 'final_story_document_coverage.csv', index=False)
